# MorphIQ SDK for Python — Option pricing

Value a call or put, compare assumptions, and examine sensitivity to the underlying price.

**Run All once to open the workspace**, then use the controls below. Load a ticker's
latest completed Yahoo close, or enter a price manually. Choose European or American
exercise, the pricing model, strike, time to expiry, volatility, interest and dividend yield.

**Price option** makes one API request. **Spot sensitivity** makes nine sequential
requests across ±20% of your underlying price, holding the other inputs fixed.
Each action may consume API credits. Changing controls and loading Yahoo data make
no MorphIQ requests. Completed valuations stay in the comparison table for this session.

Docker includes everything required. Outside Docker, install this repository with
`pip install -e '.[notebooks]'`. Set `MORPHIQ_API_KEY` before starting, or enter it
at the masked prompt when running the workspace. [Get API access](https://developers.morphiqlabs.com).

[Back to start](../start-here.ipynb)


In [ ]:
import math
import os
from datetime import date, timedelta
from getpass import getpass

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from IPython.display import display

from morphiq_sdk import APIError, MorphIQ, PriceRequest, ResponseError, TransportError

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

style = {"description_width": "140px"}
field_layout = widgets.Layout(width="350px")
ticker = widgets.Text(
    value="SPY", description="Underlying ticker", style=style, layout=field_layout
)
load_button = widgets.Button(description="Load Yahoo close", icon="download")
market_output = widgets.Output()
exercise = widgets.Dropdown(
    options=[("European", "european"), ("American", "american")],
    description="Exercise style",
    style=style,
    layout=field_layout,
)
option_type = widgets.Dropdown(
    options=[("Call", "call"), ("Put", "put")],
    description="Option type",
    style=style,
    layout=field_layout,
)
model = widgets.Dropdown(
    options=[("Automatic", "auto"), ("Black–Scholes–Merton", "black_scholes_merton")],
    description="Pricing model",
    style=style,
    layout=field_layout,
)
spot = widgets.FloatText(
    value=100.0, description="Underlying price", style=style, layout=field_layout
)
strike = widgets.FloatText(value=100.0, description="Strike", style=style, layout=field_layout)
expiry = widgets.FloatText(
    value=1.0, description="Expiry (years)", style=style, layout=field_layout
)
volatility = widgets.FloatText(
    value=20.0, description="Volatility (%)", style=style, layout=field_layout
)
rate = widgets.FloatText(
    value=5.0, description="Risk-free rate (%)", style=style, layout=field_layout
)
yield_input = widgets.FloatText(
    value=0.0, description="Dividend yield (%)", style=style, layout=field_layout
)
price_button = widgets.Button(
    description="Price option · 1 request",
    button_style="primary",
    layout=widgets.Layout(width="250px"),
)
sweep_button = widgets.Button(
    description="Spot sensitivity · 9 requests", layout=widgets.Layout(width="280px")
)
result_output = widgets.Output()
sweep_output = widgets.Output()
price_history = []
sweep_results = []
last_response = None
spot_source = "Manual input"
controls = [
    ticker,
    load_button,
    exercise,
    option_type,
    model,
    spot,
    strike,
    expiry,
    volatility,
    rate,
    yield_input,
    price_button,
    sweep_button,
]


def update_models(change):
    model.options = (
        [("Automatic", "auto"), ("Black–Scholes–Merton", "black_scholes_merton")]
        if exercise.value == "european"
        else [
            ("Automatic", "auto"),
            ("Bjerksund–Stensland 2002", "bjerksund_stensland_2002"),
            ("Bjerksund–Stensland 1993", "bjerksund_stensland_1993"),
        ]
    )
    model.value = "auto"


def mark_manual(change):
    global spot_source
    spot_source = "Manual input"


exercise.observe(update_models, names="value")
spot.observe(mark_manual, names="value")


def use_yahoo_close(history, symbol):
    global spot_source
    if history is None or history.empty:
        raise ValueError(
            "Yahoo returned no price history. Check the ticker or enter a price manually."
        )
    value = float(history["Close"].iloc[-1])
    if not math.isfinite(value) or value <= 0:
        raise ValueError("The latest Yahoo close is missing or invalid; enter a price manually.")
    spot.value = value
    spot_source = f"{symbol}: Yahoo daily close on {history.index[-1].date()}"
    print(f"{spot_source}: {value:.6g}. This is a completed daily close, not a live quote.")
    print("Strike, volatility, rates and yield remain your assumptions.")


def report_error(error):
    if isinstance(error, APIError):
        charge = error.metadata.fcu_charged if error.metadata else None
        print(f"HTTP {error.status_code}: {error.code}: {error.message}")
        print(
            f"Request ID: {error.request_id}; "
            f"CU charged: {charge if charge is not None else 'not reported'}"
        )
    elif isinstance(error, (TransportError, ResponseError)):
        print("No usable response. Execution and charges may be unknown; no retry was made.")
        if isinstance(error, ResponseError):
            print("Request ID:", error.request_id)
    else:
        print(str(error))


def load_close(_):
    with market_output:
        market_output.clear_output(wait=True)
        load_button.disabled = True
        try:
            symbol = ticker.value.strip().upper()
            if not symbol:
                raise ValueError("Enter an underlying ticker.")
            end = date.today()
            history = yf.download(
                symbol,
                start=end - timedelta(days=14),
                end=end,
                interval="1d",
                auto_adjust=False,
                keepna=True,
                multi_level_index=False,
                progress=False,
                threads=False,
            )
            use_yahoo_close(history, symbol)
        except Exception as error:
            print("Yahoo lookup failed:", error)
        finally:
            load_button.disabled = False


def current_request():
    values = [
        spot.value,
        strike.value,
        expiry.value,
        volatility.value,
        rate.value,
        yield_input.value,
    ]
    if not all(math.isfinite(value) for value in values):
        raise ValueError("All inputs must be finite numbers.")
    if min(spot.value, strike.value, expiry.value, volatility.value) <= 0:
        raise ValueError("Underlying price, strike, expiry and volatility must be positive.")
    if yield_input.value < 0:
        raise ValueError("Dividend yield must be nonnegative.")
    return PriceRequest(
        exercise_style=exercise.value,
        option_type=option_type.value,
        spot=spot.value,
        strike=strike.value,
        time_to_expiry_years=expiry.value,
        risk_free_rate=rate.value / 100,
        dividend_yield=yield_input.value / 100,
        volatility=volatility.value / 100,
        pricing_model=model.value,
    )


def result_row(request, response):
    return {
        "Style": request.exercise_style,
        "Type": request.option_type,
        "Spot": request.spot,
        "Strike": request.strike,
        "Years": request.time_to_expiry_years,
        "Vol (%)": request.volatility * 100,
        "Rate (%)": request.risk_free_rate * 100,
        "Yield (%)": request.dividend_yield * 100,
        "Price": response.result.price,
        "Model": response.result.pricing_model,
        "Version": response.result.model_version,
        "CU charged": response.metadata.fcu_charged
        if response.metadata.fcu_charged is not None
        else "not reported",
        "Request ID": response.metadata.request_id,
    }


def set_busy(busy):
    for control in controls:
        control.disabled = busy


def price_selected(_):
    global last_response
    last_response = None
    with result_output:
        result_output.clear_output(wait=True)
        set_busy(True)
        try:
            request = current_request()
            with MorphIQ() as client:
                response = client.price(request)
            last_response = response
            price_history.append(
                {"Underlying source": spot_source, **result_row(request, response)}
            )
            print(f"Option value: {response.result.price:.8g} per share, in the input currency.")
            print("No contract multiplier applied. Completed valuations:")
            display(pd.DataFrame(price_history))
        except (APIError, TransportError, ResponseError, ValueError) as error:
            report_error(error)
        finally:
            set_busy(False)


def sweep_spot(_):
    global sweep_results
    with sweep_output:
        sweep_output.clear_output(wait=True)
        sweep_results = []
        set_busy(True)
        base = None
        try:
            base = current_request()
            # Nine explicit, sequential API requests: -20% to +20% in 5% increments.
            with MorphIQ() as client:
                for step in range(-4, 5):
                    request = PriceRequest(
                        **{**base.to_wire(), "spot": base.spot * (1 + step * 0.05)}
                    )
                    response = client.price(request)
                    sweep_results.append(result_row(request, response))
        except (APIError, TransportError, ResponseError, ValueError) as error:
            print(f"Stopped after {len(sweep_results)} completed scenarios.")
            report_error(error)
        finally:
            if sweep_results:
                table = pd.DataFrame(sweep_results)
                display(table)
                fig, ax = plt.subplots(figsize=(8, 3.5))
                ax.plot(table["Spot"], table["Price"], marker="o", color="#2459a6")
                ax.set(
                    xlabel="Underlying price",
                    ylabel="Option value per share",
                    title=f"{base.exercise_style.title()} {base.option_type}: spot sensitivity",
                )
                ax.grid(alpha=0.25)
                display(fig)
                plt.close(fig)
                charges = [row["CU charged"] for row in sweep_results]
                print(
                    "Completed-scenario CU:",
                    sum(charges)
                    if all(isinstance(x, int) for x in charges)
                    else "not fully reported; see each row",
                )
            set_busy(False)


load_button.on_click(load_close)
price_button.on_click(price_selected)
sweep_button.on_click(sweep_spot)
display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Underlying</h3>"),
            widgets.HBox([ticker, load_button]),
            market_output,
            widgets.HTML("<h3>Option and assumptions</h3>"),
            widgets.HBox([exercise, option_type]),
            model,
            widgets.HBox([spot, strike]),
            widgets.HBox([expiry, volatility]),
            widgets.HBox([rate, yield_input]),
            widgets.HTML(
                "<p>Rates and volatility are percentages. Expiry is in years. "
                "Uses continuous dividend yield; cash-dividend schedules are not entered here.</p>"
            ),
            widgets.HBox([price_button, sweep_button]),
            result_output,
            sweep_output,
        ]
    )
)

### Reading the analysis

Prices are per share in the same currency as the underlying and strike; no contract
multiplier is applied. Yahoo supplies a completed daily close, not a live quote.
Volatility, rates and continuous dividend yield are your assumptions. Enter 20 for 20%.
Expiry is expressed in years. Cash-dividend schedules are outside these controls.

The sensitivity chart contains separate model valuations, not forecasts or quoted
market option prices. Each row records the inputs, resolved model, request ID and
reported commercial CU. A missing charge is unreported, not zero. If a request fails,
the sweep stops and retains completed results; no requests are retried automatically.

Results retain their original inputs when controls change. Clear outputs before sharing;
shut down the kernel when finished. Source code is available by expanding the hidden cell.
